# Datasets でデータを読み込む・加工する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_transformers/datasets_basics.ipynb)

NLP クックブック 12-10 ／ Transformers の基本操作

Hugging Face Hub のデータセットを読み込んで、フィルタ・変換・分割などの基本操作をしたい。

In [1]:
!pip install -q datasets

In [2]:
from datasets import load_dataset

ds = load_dataset("stanfordnlp/imdb")
print(ds)
print(ds["train"].features)
print(ds["train"][0]["text"][:100], "| label:", ds["train"][0]["label"])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it w | label: 0


In [3]:
train = ds["train"].shuffle(seed=0).select(range(2000))                      # シャッフルして先頭 2000 件
train = train.filter(lambda x: len(x["text"]) < 2000)                         # 条件で絞る
train = train.map(lambda x: {"n_words": len(x["text"].split())})              # 列を追加
train = train.map(lambda b: {"text": [t.replace("<br />", " ") for t in b["text"]]}, batched=True)   # バッチで変換 (速い)
print(train)

split = train.train_test_split(test_size=0.1, stratify_by_column="label", seed=0)   # 層化分割
print({k: len(v) for k, v in split.items()}, "ラベル名:", split["train"].features["label"].names)

Dataset({
    features: ['text', 'label', 'n_words'],
    num_rows: 1665
})
{'train': 1498, 'test': 167} ラベル名: ['neg', 'pos']


In [4]:
df = split["train"].to_pandas()                 # pandas へ
print(df.groupby("label")["n_words"].mean())

from datasets import Dataset
mine = Dataset.from_dict({"text": ["良い", "悪い"], "label": [1, 0]})   # 自分のデータから作る
mine.to_json("mine.jsonl", force_ascii=False)                            # JSONL で保存
print(load_dataset("json", data_files="mine.jsonl")["train"][0])

label
0    169.937984
1    165.118785
Name: n_words, dtype: float64
{'text': '良い', 'label': 1}


- `map` の結果はディスクにキャッシュされ、同じ処理は 2 回目から一瞬で終わる。
- トークナイズのような重い処理は `batched=True` で。`num_proc=2` で並列化もできる。
- 最近の `datasets` は、読み込み用の Python スクリプトを含む古い形式のデータセットに対応していない。Parquet 形式で公開されているものを選ぶ。